In [0]:
# Cell 1: Install dependencies
%pip install pmdarima statsmodels requests



In [0]:
# Cell 2: Fetch from Open-Meteo API
import requests, pandas as pd
from pyspark.sql import SparkSession

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 13.08, "longitude": 80.27,
    "start_date": "2022-01-01", "end_date": "2023-12-31",
    "hourly": "temperature_2m,relative_humidity_2m,precipitation,windspeed_10m",
    "timezone": "Asia/Kolkata"
}
r = requests.get(url, params=params)
data = r.json()["hourly"]



In [0]:
# Cell 3: Convert to pandas → Spark → Bronze Delta
pdf = pd.DataFrame(data)
pdf["time"] = pd.to_datetime(pdf["time"])
df_bronze = spark.createDataFrame(pdf)
df_bronze.write.format("delta").mode("overwrite").saveAsTable("weather_bronze")
print(f"Bronze rows: {df_bronze.count()}")



In [0]:
# Cell 4: Silver — resample to daily averages, drop nulls
from pyspark.sql.functions import to_date, avg, col

df_silver = (
    spark.read.table("weather_bronze")
    .withColumn("date", to_date("time"))
    .groupBy("date")
    .agg(
        avg("temperature_2m").alias("avg_temp"),
        avg("relative_humidity_2m").alias("avg_humidity"),
        avg("precipitation").alias("avg_precip"),
        avg("windspeed_10m").alias("avg_wind")
    )
    .orderBy("date")
    .dropna()
)
df_silver.write.format("delta").mode("overwrite").saveAsTable("weather_silver")
display(df_silver)